In [88]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [89]:
df = pd.read_csv("/content/train.csv")

In [90]:
df.sample(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
604,605,1,1,"Homer, Mr. Harry (""Mr E Haven"")",male,35.0,0,0,111426,26.5500,NaN,C
248,249,1,1,"Beckwith, Mr. Richard Leonard",male,37.0,1,1,11751,52.5542,D35,S
852,853,0,3,"Boulos, Miss. Nourelain",female,9.0,1,1,2678,15.2458,NaN,C
752,753,0,3,"Vande Velde, Mr. Johannes Joseph",male,33.0,0,0,345780,9.5000,NaN,S
881,882,0,3,"Markun, Mr. Johann",male,33.0,0,0,349257,7.8958,NaN,S
478,479,0,3,"Karlsson, Mr. Nils August",male,22.0,0,0,350060,7.5208,NaN,S
508,509,0,3,"Olsen, Mr. Henry Margido",male,28.0,0,0,C 4001,22.5250,NaN,S
154,155,0,3,"Olsen, Mr. Ole Martin",male,NaN,0,0,Fa 265302,7.3125,NaN,S
445,446,1,1,"Dodge, Master. Washington",male,4.0,0,2,33638,81.8583,A34,S
30,31,0,1,"Uruchurtu, Don. Manuel E",male,40.0,0,0,PC 17601,27.7208,NaN,C


In [91]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [92]:
df.shape

(891, 12)

### Conclusion:
  - One thing Cabin 687 values missing, where total 891 rows here so I can remove that column because morethan 50% data messing.
  - Embarked 2 missing values, and age 177 missing values.
  - Not required PassengerId, Name

In [93]:
df = df.drop(columns=['PassengerId', 'Cabin', 'Name', 'Ticket'])

In [94]:
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
31,1,1,female,NaN,1,0,146.5208,C
683,0,3,male,14.0,5,2,46.9000,S
850,0,3,male,4.0,4,2,31.2750,S
152,0,3,male,55.5,0,0,8.0500,S
578,0,3,female,NaN,1,0,14.4583,C


In [95]:
df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [97]:
df.sample(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
218,1,1,female,32.0,0,0,76.2917,C
830,1,3,female,15.0,1,0,14.4542,C


### Conclusion
  - Here Survived will be my target column.
  - Sex and Embarked will be my catagorical data column, I'll have to check what are the types present in each column.
  - Ticket is mixed type column where intiger and string both present.
  -

In [98]:
# df['Sex'].unique() # => array(['male', 'female'], dtype=object)
df['Embarked'].unique() # => Nan represent missing values.

array(['S', 'C', 'Q', nan], dtype=object)

#### Before start preprocessing my data I should be make partision on these train and test dataset.

In [99]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [100]:
X_train.sample(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
264,3,female,NaN,0,0,7.75,Q
658,2,male,23.0,0,0,13.00,S


## My Flow will be
  - 1st Handle missing values
  - 2nd Convert encoding
  - 3rd Scaling
  - 4th Feature Selection
  - 5th Train the model(Decision Tree Classifier)

#### 1st I am trying to fix missing values, in Age and Embarked columns.

In [101]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [102]:
trn_missing_age_embarked = ColumnTransformer(transformers=[
    ('age', SimpleImputer(), [2]),
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])
], remainder='passthrough')

In [103]:
from sklearn.preprocessing import OneHotEncoder

In [104]:
trn_ohe_encoder = ColumnTransformer(transformers=[
    # After trn_missing_age_embarked, columns are: Age (0), Embarked (1), Pclass (2), Sex (3), SibSp (4), Parch (5), Fare (6)
    ('one_hotencoder', OneHotEncoder(drop='first', sparse_output=False), [3, 1]) # Apply OHE to Sex (new index 3) and Embarked (new index 1)
], remainder='passthrough')

In [105]:
# Scaling Normalization -> Min Max Normalization
from sklearn.preprocessing import MinMaxScaler

In [106]:
trn_scaling = ColumnTransformer(transformers=[
    ('min_max_scaler', MinMaxScaler(), slice(0,8))
], remainder='passthrough')

In [107]:
from sklearn.feature_selection import SelectKBest, chi2

In [108]:
# Feature selection
trf4_features_selection = SelectKBest(score_func=chi2,k=8)

In [109]:
from sklearn.tree import DecisionTreeClassifier

In [110]:
trn_model = DecisionTreeClassifier()

### Pipeline


In [111]:
from sklearn.pipeline import Pipeline

In [112]:
pipe = Pipeline([
    ('missing_values', trn_missing_age_embarked),
    ('ohe_encoder', trn_ohe_encoder),
    ('scaling', trn_scaling),
    ('feature_selection', trf4_features_selection),
    ('model', trn_model)
])

In [114]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('missing_values',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('age', SimpleImputer(), [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('ohe_encoder',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('one_hotencoder',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  [3, 1])])),
                ('scaling',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('min_max_scaler',
                                                  MinMaxScaler(),
                                                  slice(0, 8, None))])),
                ('feature_selection',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7f51330c4400>)),
                ('model', DecisionTreeClassifier())])

In [115]:
# Display Pipeline

from sklearn import set_config
set_config(display='diagram')

In [116]:
pipe

Pipeline(steps=[('missing_values',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('age', SimpleImputer(), [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('ohe_encoder',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('one_hotencoder',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  [3, 1])])),
                ('scaling',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('min_max_scaler',
                                                  MinMaxScaler(),
                                                  slice(0, 8, None))])),
                ('feature_selection',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7f51330c4400>)),
                ('model', DecisionTreeClassifier())])

In [117]:
# Predict
y_pred = pipe.predict(X_test)

In [119]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.776536312849162

### Cross Validation

In [120]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()


np.float64(0.7514626218851571)

### GridSearch using Pipeline

In [122]:
from sklearn.model_selection import GridSearchCV

In [125]:
params = {
    'model__max_depth':[1,2,3,4,5,None]
}

In [126]:
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('missing_values',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('ohe_encoder',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('one_hotencoder',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         [3,
                                                                          1])])),
                                       ('scaling',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('min_max_scaler',
                                                                         MinMaxScaler(),
                                                                         slice(0, 8, None))])),
                                       ('feature_selection',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x7f51330c4400>)),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={'model__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [127]:
grid.best_score_

np.float64(0.821609376538954)

In [128]:
grid.best_params_

{'model__max_depth': 3}

### Exporting the Pipeline

In [129]:
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))